# PMAG (Protein Metal Affinity GCN)

This project is aimed at calculating the metal affinity for proteins in various bacteria. It uses the BindingDB database and a Graph Convolutional Network. Each molecule is expressed as a graph where the edges are molecules bonded to each other and each data point is a molecule.

## Processing Data

The .sdf file first has to be turned into a graph with the features, where the graph is a torch_geometric.data.Data object, containing the node and edge tables. The node features are

- element = a one-hot encoding of the element
- hybridization = the angle & shape of the bond the atom prefers
- charge = the formal charge of the atom

The edge features are

- bond type = how strong the bond is
- aromaticity = whether the electrons can change between the atoms

In [ ]:
import joblib
# Calculating a feature - replace GetHybridization() with whatever
from rdkit import Chem

def grab_first_valid(mols):
    for mol in mols:
        if mol != None: return mol

path_mols = 'data/train_input.csv'
supplier = Chem.SmilesMolSupplier(path_mols, delimiter=',', smilesColumn=0, nameColumn=-1, titleLine=True)
print('Loaded')

mol = grab_first_valid(supplier)
Chem.SanitizeMol(mol)

for atom in mol.GetAtoms():
    print(atom.GetHybridization())

In [8]:
# Generating one-hot encodings
from rdkit import Chem
from joblib import dump

path_mols = 'data/train_input.csv'
supplier = Chem.SmilesMolSupplier(path_mols, delimiter=',', smilesColumn=0, nameColumn=-1, titleLine=True)

ids = {}

for mol in supplier:
    if mol == None: continue

    for atom in mol.GetAtoms():
        if atom.GetSymbol() in ids:
            pass
        else:
            ids[atom.GetSymbol()] = len(ids) + 1

encodings = {}

for i, id in enumerate(ids.items()):
    vector_encoded = [0] * len(ids)
    vector_encoded[i] = 1
    encodings[id[0]] = vector_encoded

dump(encodings, 'data/one-hot.joblib')

[21:49:28] Unusual charge on atom 1 number of radical electrons set to zero
[21:49:28] Unusual charge on atom 1 number of radical electrons set to zero
[21:49:28] Unusual charge on atom 1 number of radical electrons set to zero
[21:49:28] Unusual charge on atom 1 number of radical electrons set to zero
[21:49:28] Unusual charge on atom 7 number of radical electrons set to zero
[21:49:28] Unusual charge on atom 7 number of radical electrons set to zero
[21:49:28] Unusual charge on atom 4 number of radical electrons set to zero
[21:49:28] Unusual charge on atom 4 number of radical electrons set to zero


['data/one-hot.joblib']

In [ ]:
# Graph creation code - run workers.py instead because of IPython multiprocessing issues
import numpy as np
import torch
from torch_geometric.data import Data
import os
import multiprocessing as mp
from rdkit import Chem
from joblib import load

chunk_size = 200

def get_chunks(file_path, chunk_size):
    chunk = []
    supplier = Chem.SmilesMolSupplier(file_path, delimiter=',', smilesColumn=0, nameColumn=-1, titleLine=True)

    for mol in supplier:
        if mol is None: continue
        chunk.append(mol)

        if len(chunk) == chunk_size:
            yield chunk
            chunk = []

encodings = load('./data/one-hot.joblib')

def process(chunk):
    chunk_to_save = []
    for mol in chunk:
        X = []
        edge_features = []

        edge_matrix_idx1 = []
        edge_matrix_idx2 = []

        for atom in mol.GetAtoms():
            X.append([*encodings[atom.GetSymbol()], int(atom.GetHybridization()), atom.GetFormalCharge()])

        for bond in mol.GetBonds():
            features_for_edge = [bond.GetBondType().real, float(bond.GetIsAromatic())]
            edge_features.append(features_for_edge)
            edge_features.append(features_for_edge)

            edge_matrix_idx1.append(bond.GetBeginAtomIdx())
            edge_matrix_idx2.append(bond.GetEndAtomIdx())

            edge_matrix_idx2.append(bond.GetBeginAtomIdx())
            edge_matrix_idx1.append(bond.GetEndAtomIdx())

        edge_matrix = torch.from_numpy(np.array([edge_matrix_idx1, edge_matrix_idx2]))
        X = torch.from_numpy(np.array(X))
        edge_features = torch.from_numpy(np.array(edge_features))
        graph = Data(x=X, edge_index=edge_matrix, edge_attr=edge_features)

        chunk_to_save.append(graph)
        if len(chunk_to_save) == chunk_size:
            if len(os.listdir('data/graph_chunks')) > 0:
                next_to_save = max([int(filename[:4]) + 1 for filename in os.listdir('data/graph_chunks')])
            else:
                next_to_save = 1
            torch.save(chunk_to_save, f'./data/graph_chunks/{next_to_save:04d}.pt')

if __name__ == '__main__':
    mols_path = 'data/train_input.csv'

    with mp.Pool(processes=5) as pool:
        results = pool.imap(process, get_chunks(mols_path, chunk_size))
        for _ in results: pass

## Training the Model

The model is a graph convolutional network with message passing in pytorch.

In [ ]:
import lightning as L
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, global_add_pool


class PMAG(nn.Module):
    def __init__(self, node_dim, edge_dim):
        super().__init__()

        model = []

        for part in range(4):
            layer = nn.Sequential(
                nn.Linear(node_dim, 128),
                nn.ReLU(),
                nn.Linear(128, 128),
                nn.ReLU(),
            )

            model.append(layer)

        self.conv = GINEConv(nn=model, train_eps=True, edge_dim=edge_dim)
        self.head = nn.Linear(64, 1)

    def forward(self, data):
        X = F.relu(self.conv(data.x, data.edge_index, data.edge_attr))
        embeds = global_add_pool(X, data.batch)
        return self.head(embeds)

    def training_step(self, batch, batch_idx):
        pred = self(batch)
        loss = F.mse_loss(pred, batch)
        self.log("Loss:", loss)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)

